# ClearPath: NLP and computer vision analysis

Supporting notebook for the report. It shows, step by step, the preprocessing and representation of the texts, what the classifier learned, where it fails, and each filter of the computer vision pipeline.

**Data:** `reports_train.csv` is **synthetic** (generated by `src/backend/scripts/generate_dataset.py`); `reports_blind_test.csv` has 60 sentences written separately from the generator templates (never used in training); the sidewalk images are **synthetic** (`generate_demo_images.py`).

In [1]:
import sys, pathlib
# repository root = first folder (cwd or above) with pyproject.toml; works whatever the kernel's working dir is
REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
ROOT = REPO / 'src'                # source code and data
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'backend' / 'scripts'))
import numpy as np, pandas as pd, cv2
import chart_style
plt = chart_style.apply()
from clearpath import nlp, governance, vision
train = pd.read_csv(ROOT / 'data' / 'reports_train.csv')
blind = pd.read_csv(ROOT / 'data' / 'reports_blind_test.csv')
print(train.shape, blind.shape)
train.category.value_counts()

(1320, 3) (60, 3)


category
crossing_signal         220
sidewalk_obstruction    220
tactile_paving          220
accessible_transit      220
ramp_access             220
damaged_sidewalk        220
Name: count, dtype: int64

## 1. Governance first: anonymization
The text is anonymized **before** it reaches the model and the database. How many synthetic reports contain personal data?

In [2]:
anon = train.text.map(governance.anonymize)
train['has_pii'] = anon.map(lambda r: r.had_personal_data)
print(f"{train.has_pii.mean():.0%} of the training reports contain personal data inserted on purpose")
example = train[train.has_pii].text.iloc[0]
print('ORIGINAL  :', example)
print('ANONYMIZED:', governance.anonymize(example).text)

43% of the training reports contain personal data inserted on purpose
ORIGINAL  : Hello, crossing with no refuge island, the avenue is too wide, my mother is elderly and uses a walker, call me at (11) 99924-7887 I ask for action.
ANONYMIZED: Hello, crossing with no refuge island, the avenue is too wide, my mother is elderly and uses a walker, call me at [PHONE] I ask for action.


## 2. Preprocessing
Lowercase → no accents → no markers/digits/punctuation → tokens without stopwords.

In [3]:
t = 'URGENT!! The sidewalk on Rua Augusta no. 1500 is all broken, my elderly mother fell. cpf 123.456.789-09'
a = governance.anonymize(t).text
print('1 anonymized :', a)
print('2 normalized :', nlp.normalize(a))
print('3 tokens     :', nlp.tokenize(a))

1 anonymized : URGENT!! The sidewalk on Rua Augusta no. 1500 is all broken, my elderly mother fell. cpf [CPF]
2 normalized : urgent the sidewalk on rua augusta no is all broken my elderly mother fell cpf
3 tokens     : ['urgent', 'sidewalk', 'rua', 'augusta', 'broken', 'elderly', 'mother', 'fell', 'cpf']


In [4]:
from collections import Counter
lengths = train.text.map(lambda x: len(nlp.tokenize(governance.anonymize(x).text)))
print('tokens per report: mean', round(lengths.mean(), 1), '| min', lengths.min(), '| max', lengths.max())
vocab = Counter(tok for x in train.text for tok in nlp.tokenize(governance.anonymize(x).text))
print('vocabulary:', len(vocab), 'terms | most frequent:', vocab.most_common(12))

tokens per report: mean 13.1 | min 5 | max 21
vocabulary: 522 terms | most frequent: [('number', 427), ('sidewalk', 370), ('walk', 352), ('wheelchair', 337), ('get', 266), ('rua', 248), ('ramp', 229), ('tactile', 213), ('front', 211), ('uses', 205), ('av', 202), ('before', 202)]


## 3. Representation and what the model learned
Word TF-IDF (1-2 grams) + character n-gram TF-IDF (3-5). Below, the word terms with the highest Logistic Regression weight for each category.

In [5]:
import joblib
from clearpath import config
model = joblib.load(config.MODEL_FILE)
union = model.named_steps['tfidf']
word_vec = dict(union.transformer_list)['words']
n_words = len(word_vec.vocabulary_)
terms = np.array(word_vec.get_feature_names_out())
coef = model.named_steps['clf'].coef_[:, :n_words]
pd.DataFrame({nlp.CATEGORIES[c]: terms[np.argsort(-coef[i])[:8]] for i, c in enumerate(model.classes_)})

,Accessible public transit,Crossing and pedestrian signal,Damaged sidewalk,Ramp / building access,Sidewalk obstruction,Tactile paving
0,bus,crosswalk,sidewalk,ramp,parked,tactile
1,elevator,light,step,curb,sidewalk,paving
2,braille signage,pedestrian,sunken,hole middle,blocking,tactile paving
3,braille,faded,sidewalk signage,ramp hole,car,path
4,station braille,signal,huge hole,ramp get,car parked,strip
5,driver,faded crosswalk,pavement,ramp narrow,sitting,tactile strip
6,accessible,traffic,hole sidewalk,narrow wheelchair,sitting sidewalk,paving crosswalk
7,accessible sidewalk,button,step middle,onto sidewalk,dumpster sitting,guide


## 4. Honest evaluation: blind test
In cross-validation on synthetic data the F1 is ~1.00 (the sentences come from the same templates). On the hand-written blind test it drops to ~0.90. **This gap is the main limitation of the NLP component.**

Naive Bayes and Logistic Regression tied on the blind test (difference of 0.0004 F1, same 90% accuracy). The selection rule treats differences below 0.01 as a tie, broken by the cross-validation F1, which picks Logistic Regression.

In [6]:
pd.read_csv(REPO / 'document' / 'other' / 'evidence' / 'nlp_model_comparison.csv')

,model,f1_cv,f1_cv_std,f1_blind_test,accuracy_blind_test
0,Naive Bayes (words),0.9977,0.0045,0.8991,0.9000
1,Logistic Regression (words + characters),1.0000,0.0000,0.8987,0.9000
2,Calibrated linear SVM (words + characters),1.0000,0.0000,0.8814,0.8833


In [7]:
cls = [nlp.classify(governance.anonymize(x).text) for x in blind.text]
blind['predicted'] = [c.category for c in cls]
blind['confidence'] = [c.confidence for c in cls]
blind['correct'] = blind.predicted == blind.category
print('accuracy:', round(blind.correct.mean(), 3))
blind[~blind.correct][['text', 'category', 'predicted', 'confidence']]

accuracy: 0.9


,text,category,predicted,confidence
1,I tripped on a raised concrete slab near the c...,damaged_sidewalk,crossing_signal,0.629
13,there is a pile of sand and bricks from a reno...,sidewalk_obstruction,tactile_paving,0.823
19,Street vendors occupy the entire sidewalk at t...,sidewalk_obstruction,damaged_sidewalk,0.629
32,There is no floor signage for cane users in th...,tactile_paving,accessible_transit,0.439
39,Pieces of the directional tiles are missing on...,tactile_paving,ramp_access,0.847
42,"The zebra stripes are gone from wear, drivers ...",crossing_signal,accessible_transit,0.647


### Confidence threshold for human triage
How much accuracy do we gain by forwarding automatically only what passes the threshold, and how many reports go to a public servant?

In [8]:
rows = []
for th in [0.0, 0.4, 0.5, 0.6, 0.7, 0.8]:
    auto = blind[blind.confidence >= th]
    rows.append({'threshold': th, 'auto_forwarded': len(auto), 'human_triage': len(blind) - len(auto),
                 'accuracy_of_auto': round(auto.correct.mean(), 3), 'errors_reaching_agency': int((~auto.correct).sum())})
pd.DataFrame(rows)

,threshold,auto_forwarded,human_triage,accuracy_of_auto,errors_reaching_agency
0,0.0,60,0,0.900,6
1,0.4,59,1,0.898,6
2,0.5,57,3,0.912,5
3,0.6,53,7,0.906,5
4,0.7,45,15,0.956,2
5,0.8,40,20,0.950,2


We chose **0.50**: only 5% of the reports need a public servant, and meaningless texts (confidence ~0.30-0.36) are held back. A higher threshold (0.70) would stop more errors but would send 25% of the reports to manual triage.

## 5. Explainable urgency

In [9]:
for t in ['broken sidewalk on the street', 'broken sidewalk, my elderly mother fell in front of the hospital, it has been like this for months']:
    print(nlp.score_urgency(t, 'damaged_sidewalk'), '<-', t)

(20, []) <- broken sidewalk on the street
(82, ['fall or accident reported', 'vulnerable person affected', 'near an essential service', 'recurring problem']) <- broken sidewalk, my elderly mother fell in front of the hospital, it has been like this for months


## 6. Computer vision step by step
Synthetic image from the test set with cracks, a pothole and paving joints.

In [10]:
img = cv2.imread(str(ROOT / 'data' / 'demo_images' / 'test' / 'severe_03.jpg'))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
smooth = cv2.GaussianBlur(clahe, (5, 5), 0)
blackhat = cv2.morphologyEx(smooth, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)))
cracks, potholes, n_joints, base, comp = vision.segment(img)
analysis, overlay = vision.analyze(img)
steps = [('1. Original', cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), ('2. Grayscale + CLAHE', clahe), ('3. Gaussian + black-hat', blackhat),
         (f'4. Cracks (joints removed: {n_joints})', cracks), ('5. Potholes (17 px opening)', potholes), ('6. Result', cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (title, im) in zip(axes.flat, steps):
    ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=10); ax.axis('off')
fig.tight_layout(); fig.savefig(REPO / 'assets' / 'charts' / 'vision_steps.png', dpi=130)
print(analysis)

ImageAnalysis(damage_index=42.7, level='severe', crack_pct=np.float64(3.54), pothole_pct=np.float64(1.81), joints_removed=29, crack_components=11)


## 7. Result on the image test set (synthetic, 45 images)

In [11]:
import json
m = json.loads((REPO / 'document' / 'other' / 'evidence' / 'vision_metrics.json').read_text(encoding='utf-8'))
print('accuracy:', m['test_accuracy'], '| thresholds:', m['thresholds'])
pd.DataFrame(m['confusion_matrix']['values'], index=['true: ' + r for r in m['confusion_matrix']['labels']],
             columns=['pred: ' + r for r in m['confusion_matrix']['labels']])

accuracy: 0.8667 | thresholds: {'moderate': 6, 'severe': 27}


,pred: no visible damage,pred: moderate,pred: severe
true: no visible damage,13,2,0
true: moderate,3,12,0
true: severe,0,1,14


**Limitation:** synthetic images are cleaner than real photos (shadows, leaves, oil stains, Portuguese cobblestones and tactile paving may cause false positives). That is why the damage index only **reinforces** priority; it never decides on its own.